# Data Engineering for LLM Fine-Tuning

This notebook provides hands-on examples for data engineering workflows covered in Module 04. We'll cover:

1. Loading datasets from Hugging Face and JSON
2. Quality filtering (length, language, toxicity)
3. Exact and fuzzy deduplication with MinHash
4. ChatML formatting
5. Train/validation/test splitting
6. Tokenization and preprocessing
7. Complete preprocessing pipeline

## Setup and Installation

In [ ]:
# Install required packages (uncomment if needed)
# !pip install datasets langdetect minhash textblob pandas tqdm
# !pip install transformers accelerate peft

import json
import hashlib
import re
from typing import List, Dict, Any
from collections import Counter

from tqdm import tqdm
import pandas as pd
import numpy as np

## 1. Loading Datasets

### Loading from Hugging Face Hub

In [ ]:
from datasets import load_dataset

# Load a popular instruction dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:1000]")

print(f"Loaded {len(dataset)} samples")
print(f"Columns: {dataset.column_names}")
print(f"\nSample entry:")
print(dataset[0])

### Loading from JSON/JSONL Files

In [ ]:
# Example: Loading from JSONL (common format for fine-tuning)
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file into list of dictionaries."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Example: Loading from JSON
def load_json(filepath: str) -> List[Dict]:
    """Load JSON file into list of dictionaries."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data if isinstance(data, list) else [data]

# Create sample data for demonstration
sample_data = [
    {"instruction": "Write a haiku about coding", "input": "", "output": "Lines of code flow free / Logic dances in the screen / Bugs hide, then emerge"},
    {"instruction": "Explain recursion", "input": "", "output": "Recursion is when a function calls itself. It needs a base case to stop, otherwise it runs forever. Think of Russian nesting dolls."},
    {"instruction": "Convert to JSON", "input": "name: John, age: 30", "output": '{"name": "John", "age": 30}'}
]

# Save as JSONL for demonstration
with open('sample_data.jsonl', 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')

# Load it back
loaded_data = load_jsonl('sample_data.jsonl')
print(f"Loaded {len(loaded_data)} samples from JSONL")
print(json.dumps(loaded_data[0], indent=2))

## 2. Quality Filtering

### Length-Based Filtering

In [ ]:
def filter_by_length(data: List[Dict], 
                     min_chars: int = 50,
                     max_chars: int = 8192,
                     min_words: int = 10,
                     text_field: str = 'output') -> List[Dict]:
    """Filter samples by character and word count."""
    filtered = []
    for item in data:
        text = item.get(text_field, '')
        char_count = len(text)
        word_count = len(text.split())
        
        if char_count >= min_chars and char_count <= max_chars and word_count >= min_words:
            filtered.append(item)
    
    removed = len(data) - len(filtered)
    print(f"Removed {removed}/{len(data)} samples ({100*removed/len(data):.1f}%) by length")
    return filtered

# Demonstrate with extended sample data
extended_data = sample_data + [
    {"instruction": "Too short", "input": "", "output": "OK"},
    {"instruction": "Good sample", "input": "", "output": "This is a properly sized response that meets the minimum requirements for words and characters."}
]

filtered = filter_by_length(extended_data, min_chars=30, min_words=5)
print(f"\nKept {len(filtered)} samples")

### Language Detection and Filtering

In [ ]:
try:
    from langdetect import detect, DetectorFactory
    from langdetect.lang_detect_exception import LangDetectException
    
    # Set seed for reproducibility
    DetectorFactory.seed = 0
    
    def filter_by_language(data: List[Dict], 
                           target_lang: str = 'en',
                           text_field: str = 'output') -> List[Dict]:
        """Filter to keep only samples in target language."""
        filtered = []
        lang_counts = Counter()
        
        for item in tqdm(data, desc="Detecting language"):
            text = item.get(text_field, '')
            if len(text) < 10:  # Too short to detect reliably
                continue
            try:
                lang = detect(text)
                lang_counts[lang] += 1
                if lang == target_lang:
                    filtered.append(item)
            except LangDetectException:
                continue
        
        print(f"Language distribution: {dict(lang_counts)}")
        print(f"Kept {len(filtered)}/{len(data)} samples in {target_lang}")
        return filtered
    
    # Example usage (uncomment to run)
    # filtered_en = filter_by_language(extended_data, target_lang='en')
    
except ImportError:
    print("langdetect not installed. Install with: pip install langdetect")

### Toxicity Detection

In [ ]:
def simple_toxicity_filter(data: List[Dict], 
                            text_fields: List[str] = None,
                            threshold: float = 0.5) -> List[Dict]:
    """
    Filter toxic content using keyword-based approach.
    For production, use perspective API or detoxify library.
    """
    if text_fields is None:
        text_fields = ['output', 'response', 'answer']
    
    # Simple keyword list (expand for production use)
    toxic_keywords = [
        'stupid', 'idiot', 'hate', 'kill', 'die',
        'stupid', 'idiot', 'hate', 'kill', 'die',
    ]
    
    filtered = []
    toxic_count = 0
    
    for item in data:
        is_toxic = False
        for field in text_fields:
            text = item.get(field, '').lower()
            if any(kw in text for kw in toxic_keywords):
                is_toxic = True
                break
        
        if not is_toxic:
            filtered.append(item)
        else:
            toxic_count += 1
    
    print(f"Removed {toxic_count}/{len(data)} potentially toxic samples")
    return filtered

# For production-grade toxicity detection:
# !pip install detoxify
# from detoxify import Detoxify
# model = Detoxify('original')
# results = model.predict("Your text here")

### PII Removal

In [ ]:
def remove_pii(text: str) -> str:
    """Remove personally identifiable information using regex patterns."""
    
    # Email addresses
    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL]', text)
    
    # Phone numbers (various formats)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\+?\d[\d\s\-]{8,}\d', '[PHONE]', text)
    
    # Social Security Numbers
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    
    # Credit card numbers
    text = re.sub(r'\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b', '[CC]', text)
    
    # IP addresses
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '[IP]', text)
    
    return text

# Test PII removal
test_text = """Contact john.doe@example.com or call 555-123-4567.
His SSN is 123-45-6789 and IP is 192.168.1.1"""

cleaned = remove_pii(test_text)
print("Original:", test_text)
print("\nCleaned:", cleaned)

## 3. Deduplication

### Exact Deduplication

In [ ]:
def exact_dedup(data: List[Dict], text_field: str = 'output') -> List[Dict]:
    """Remove exact duplicates based on text content."""
    seen = set()
    deduped = []
    
    for item in data:
        text = item.get(text_field, '')
        text_hash = hashlib.md5(text.encode()).hexdigest()
        
        if text_hash not in seen:
            seen.add(text_hash)
            deduped.append(item)
    
    removed = len(data) - len(deduped)
    print(f"Removed {removed}/{len(data)} exact duplicates ({100*removed/len(data):.1f}%)")
    return deduped

# Test with duplicates
data_with_dups = sample_data + sample_data[:2]  # Add some duplicates
deduped = exact_dedup(data_with_dups)
print(f"Before: {len(data_with_dups)}, After: {len(deduped)}")

### Fuzzy Deduplication with MinHash

In [ ]:
try:
    from datasketch import MinHash, MinHashLSH
    
    def fuzzy_dedup_minhash(data: List[Dict], 
                            text_field: str = 'output',
                            threshold: float = 0.8) -> List[Dict]:
        """
        Remove near-duplicates using MinHash and LSH.
        Samples with Jaccard similarity > threshold are considered duplicates.
        """
        
        def create_minhash(text: str, num_perm: int = 128) -> MinHash:
            """Create MinHash object for text."""
            m = MinHash(num_perm=num_perm)
            # Shingle by words
            shingles = text.lower().split()
            for shingle in shingles:
                m.update(shingle.encode('utf8'))
            return m
        
        # Build MinHash for all samples
        minhashes = []
        for item in tqdm(data, desc="Building MinHash"):
            text = item.get(text_field, '')
            if text.strip():
                minhashes.append((item, create_minhash(text)))
        
        # Use LSH for efficient similarity search
        lsh = MinHashLSH(threshold=threshold, num_perm=128)
        
        deduped = []
        duplicate_indices = set()
        
        for idx, (item, minhash) in enumerate(tqdm(minhashes, desc="Deduplicating")):
            # Check if this is similar to any previously kept item
            is_duplicate = False
            
            # Query LSH for potential duplicates
            for dup_idx in lsh.query(minhash):
                if minhash.jaccard(minhashes[dup_idx][1]) >= threshold:
                    is_duplicate = True
                    break
            
            if not is_duplicate:
                deduped.append(item)
                lsh.insert(idx, minhash)
            else:
                duplicate_indices.add(idx)
        
        removed = len(duplicate_indices)
        print(f"Removed {removed}/{len(data)} fuzzy duplicates ({100*removed/len(data):.1f}%)")
        return deduped
    
    # Install with: pip install datasketch
    # Example usage:
    # deduped_fuzzy = fuzzy_dedup_minhash(large_dataset, threshold=0.85)
    
except ImportError:
    print("datasketch not installed. Install with: pip install datasketch")

## 4. ChatML Formatting

In [ ]:
def format_chatml(conversations: List[Dict], 
                  system_prompt: str = "You are a helpful assistant.") -> str:
    """
    Format conversations in ChatML format.
    
    Args:
        conversations: List of message dicts with 'role' and 'content' keys
        system_prompt: System message to prepend
    
    Returns:
        Formatted ChatML string
    """
    chatml = f"<|system|>\n{system_prompt}<|end|>\n"
    
    for msg in conversations:
        role = msg.get('role', 'user')
        content = msg.get('content', '')
        chatml += f"<|{role}|>\n{content}<|end|>\n"
    
    # Add assistant prefix for generation
    chatml += "<|assistant|>\n"
    
    return chatml

# Example usage
sample_conversation = [
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of AI that enables systems to learn from data."},
    {"role": "user", "content": "Can you give an example?"}
]

formatted = format_chatml(sample_conversation)
print(formatted)

In [ ]:
# Convert instruction dataset to ChatML
def instruction_to_chatml(instruction: str, input: str, output: str) -> str:
    """Convert instruction/input/output format to ChatML."""
    user_content = instruction
    if input.strip():
        user_content += f"\n\nInput:\n{input}"
    
    conversations = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    return format_chatml(conversations)

# Test conversion
chatml_sample = instruction_to_chatml(
    instruction="Explain quantum computing",
    input="",
    output="Quantum computing uses quantum bits (qubits) that can exist in superposition..."
)
print(chatml_sample)

## 5. Train/Validation/Test Splitting

In [ ]:
from sklearn.model_selection import train_test_split

def split_dataset(data: List[Dict], 
                  train_ratio: float = 0.8,
                  val_ratio: float = 0.1,
                  test_ratio: float = 0.1,
                  seed: int = 42) -> Dict[str, List[Dict]]:
    """
    Split dataset into train, validation, and test sets.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.001
    
    # First split: train vs (val + test)
    train, temp = train_test_split(
        data, 
        test_size=(val_ratio + test_ratio),
        random_state=seed
    )
    
    # Second split: val vs test
    val, test = train_test_split(
        temp,
        test_size=test_ratio / (val_ratio + test_ratio),
        random_state=seed
    )
    
    print(f"Split complete: {len(train)} train, {len(val)} val, {len(test)} test")
    print(f"Ratios: {len(train)/len(data):.1%} / {len(val)/len(data):.1%} / {len(test)/len(data):.1%}")
    
    return {"train": train, "val": val, "test": test}

# Test splitting
large_sample = sample_data * 100  # Create larger dataset
splits = split_dataset(large_sample)

# Save splits to files
for split_name, split_data in splits.items():
    with open(f'data_{split_name}.jsonl', 'w') as f:
        for item in split_data:
            f.write(json.dumps(item) + '\n')
    print(f"Saved {len(split_data)} samples to data_{split_name}.jsonl")

## 6. Tokenization

In [ ]:
from transformers import AutoTokenizer

def demonstrate_tokenization(text: str, model_name: str = "mistralai/Mistral-7B-v0.1"):
    """Demonstrate tokenization with different models."""
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Tokenize
    tokens = tokenizer.encode(text)
    decoded = tokenizer.decode(tokens)
    
    print(f"Original: {text}")
    print(f"Tokens: {tokens}")
    print(f"Token count: {len(tokens)}")
    print(f"Decoded: {decoded}")
    
    # Show token breakdown
    print("\nToken breakdown:")
    for i, token in enumerate(tokens):
        print(f"  {i}: {token} -> '{tokenizer.decode([token])}'")

# Example (uncomment to run - requires model download)
# sample_text = "Hello! I'm a fine-tuned language model."
# demonstrate_tokenization(sample_text)

In [ ]:
# Batch tokenization for dataset preprocessing
def tokenize_dataset(data: List[Dict], 
                     tokenizer: AutoTokenizer,
                     text_field: str = 'output',
                     max_length: int = 512) -> List[Dict]:
    """
    Tokenize a dataset for training.
    """
    tokenized = []
    
    for item in tqdm(data, desc="Tokenizing"):
        text = item.get(text_field, '')
        
        # Tokenize with truncation
        encoding = tokenizer(
            text,
            max_length=max_length,
            truncation=True,
            padding=False,  # Don't pad for training efficiency
            return_tensors=None
        )
        
        item['input_ids'] = encoding['input_ids']
        item['attention_mask'] = encoding['attention_mask']
        item['token_count'] = len(encoding['input_ids'])
        
        tokenized.append(item)
    
    # Report statistics
    token_counts = [x['token_count'] for x in tokenized]
    print(f"\nTokenization statistics:")
    print(f"  Min tokens: {min(token_counts)}")
    print(f"  Max tokens: {max(token_counts)}")
    print(f"  Mean tokens: {np.mean(token_counts):.1f}")
    print(f"  Median tokens: {np.median(token_counts):.1f}")
    
    return tokenized

## 7. Complete Preprocessing Pipeline

In [ ]:
class DataPreprocessingPipeline:
    """
    Complete data preprocessing pipeline for LLM fine-tuning.
    """
    
    def __init__(self, tokenizer_name: str = "mistralai/Mistral-7B-v0.1"):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.stats = {}
    
    def load(self, filepath: str) -> List[Dict]:
        """Load data from JSONL file."""
        print(f"Loading data from {filepath}...")
        self.stats['loaded'] = 0
        
        data = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line.strip()))
        
        self.stats['loaded'] = len(data)
        print(f"Loaded {len(data)} samples")
        return data
    
    def filter_quality(self, data: List[Dict]) -> List[Dict]:
        """Apply quality filters."""
        print("\nApplying quality filters...")
        
        # Length filter
        filtered = filter_by_length(data, min_chars=50, min_words=10)
        
        # PII removal
        for item in filtered:
            for field in ['output', 'response', 'answer']:
                if field in item:
                    item[field] = remove_pii(item[field])
        
        self.stats['after_quality'] = len(filtered)
        return filtered
    
    def deduplicate(self, data: List[Dict]) -> List[Dict]:
        """Remove duplicates."""
        print("\nRemoving duplicates...")
        deduped = exact_dedup(data)
        self.stats['after_dedup'] = len(deduped)
        return deduped
    
    def format_chatml(self, data: List[Dict]) -> List[Dict]:
        """Convert to ChatML format."""
        print("\nConverting to ChatML format...")
        formatted = []
        
        for item in tqdm(data, desc="Formatting"):
            if 'instruction' in item:
                chatml_text = instruction_to_chatml(
                    item.get('instruction', ''),
                    item.get('input', ''),
                    item.get('output', '')
                )
                formatted.append({
                    'text': chatml_text,
                    'original': item
                })
        
        self.stats['after_format'] = len(formatted)
        return formatted
    
    def tokenize(self, data: List[Dict], max_length: int = 512) -> List[Dict]:
        """Tokenize the dataset."""
        print("\nTokenizing...")
        tokenized = []
        
        for item in tqdm(data, desc="Tokenizing"):
            encoding = self.tokenizer(
                item['text'],
                max_length=max_length,
                truncation=True,
                return_tensors=None
            )
            
            item['input_ids'] = encoding['input_ids']
            item['attention_mask'] = encoding['attention_mask']
            item['token_count'] = len(encoding['input_ids'])
            tokenized.append(item)
        
        self.stats['final'] = len(tokenized)
        return tokenized
    
    def split_and_save(self, data: List[Dict], output_dir: str = "processed_data"):
        """Split dataset and save to files."""
        import os
        os.makedirs(output_dir, exist_ok=True)
        
        print(f"\nSplitting and saving to {output_dir}...")
        splits = split_dataset(data)
        
        for split_name, split_data in splits.items():
            filepath = f"{output_dir}/{split_name}.jsonl"
            with open(filepath, 'w', encoding='utf-8') as f:
                for item in split_data:
                    # Save only essential fields
                    save_item = {
                        'text': item['text'],
                        'input_ids': item['input_ids'],
                        'attention_mask': item['attention_mask']
                    }
                    f.write(json.dumps(save_item) + '\n')
            print(f"Saved {len(split_data)} samples to {filepath}")
        
        # Save statistics
        with open(f"{output_dir}/stats.json", 'w') as f:
            json.dump(self.stats, f, indent=2)
        
        print(f"\nPipeline statistics: {self.stats}")

# Example usage:
# pipeline = DataPreprocessingPipeline()
# data = pipeline.load('raw_data.jsonl')
# data = pipeline.filter_quality(data)
# data = pipeline.deduplicate(data)
# data = pipeline.format_chatml(data)
# data = pipeline.tokenize(data)
# pipeline.split_and_save(data, 'processed_output')

## Summary

This notebook covered the complete data engineering pipeline for LLM fine-tuning:

1. **Loading**: From Hugging Face Hub and JSON/JSONL files
2. **Quality Filtering**: Length, language, toxicity, PII removal
3. **Deduplication**: Exact and fuzzy (MinHash) methods
4. **Formatting**: ChatML format for chat models
5. **Splitting**: Train/validation/test splits
6. **Tokenization**: Model-specific tokenization
7. **Pipeline**: End-to-end preprocessing class

### Key Takeaways:

- Quality filtering removes 20-40% of raw data typically
- Deduplication can reduce dataset size by 10-30%
- ChatML is the standard format for modern chat models
- Always save train/val/test splits for proper evaluation
- Tokenization statistics help identify outliers and set max_length